# EEBG2026 HyPhy Selection Tutorial

This notebook accompanies the command-line scripts in this repository. Run the shell commands from the repository root, then use the Python cells to inspect outputs, make plots, and answer the prompts.

Current repository structure:

```text
.
├── data/
│   ├── 21-empirical/              # empirical codon alignments
│   └── empirical_alignment_metrics.csv
├── docs/
│   ├── instructor_notes.md
│   └── selection_tutorial_student_handout.md
├── notebooks/
│   ├── EEBG2026_HyPhy_Selection_Tutorial.ipynb
│   └── EEBG2026_HyPhy_Selection_Tutorial_Colab.ipynb
├── pdf/
│   └── msad150.pdf
├── scripts/                       # setup, HyPhy runs, collection helpers
├── results/                       # generated HyPhy JSON outputs
├── tables/                        # generated CSV summaries
└── figures/                       # generated plots
```


## Setup

```bash
bash scripts/00_setup_conda.sh
conda activate eebg2026-hyphy
bash scripts/01_download_tutorial_data.sh
```

The setup script verifies the empirical Nexus files in `data/21-empirical` and the metrics table at `data/empirical_alignment_metrics.csv`.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 80)
ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path.cwd()
RESULTS = ROOT / 'results'
TABLES = ROOT / 'tables'
FIGURES = ROOT / 'figures'
ROOT

## Session 1: Gene-wide Selection

Run:

```bash
bash scripts/02_run_gene_wide.sh
python scripts/07_collect_results.py
```

BUSTED is a gene-wide test for episodic diversifying selection. It asks whether at least one codon site on the tested branches has experienced positive selection at some point in the tree. A significant result supports selection somewhere in the gene, but it does not identify the exact codon site.

This session compares standard BUSTED with model variants that allow synonymous-rate variation and multiple-hit nucleotide changes. It also runs FitMultiModel, which fits the standard MG94 one-hit model and compares it with MG94 models that allow double-hit and double+triple-hit codon changes.

Prompts:

- Before looking at the p-values, what kind of biological scenario would make this gene a good candidate for episodic diversifying selection?
- What biological question is BUSTED answering for this gene?
- Does a significant BUSTED result identify the selected codon site? What would you need to run next to localize the signal?
- Did synonymous-rate variation change the result enough to affect your interpretation?
- Did BUSTED multi-hit modeling or FitMultiModel change the result enough to affect your interpretation?
- Which result would you report in a manuscript, and what caveat would you include?


In [ ]:
summary1 = pd.read_csv(TABLES / 'session1_gene_wide_summary_long.csv')
summary1

In [ ]:
img = FIGURES / 'session1_gene_wide_pvalues.png'
plt.imshow(plt.imread(img))
plt.axis('off');

## Session 2: Branch-Level Selection

Run:

```bash
bash scripts/04_run_branch_lineage.sh
python scripts/07_collect_results.py
```

This session asks whether particular branches, rather than the whole tree, show evidence of episodic diversifying selection. aBSREL fits branch-specific models and then tests branches individually, so the result is a branch-level scan rather than a gene-wide yes-or-no answer.

Because many branches are tested, multiple-testing correction matters. An uncorrected low p-value can look exciting, but the corrected result is the one you should use when deciding which branches have convincing evidence.

The empirical files used here do not define named test and reference branch sets, so this session focuses on aBSREL rather than RELAX. If you later add branch labels, you could ask different lineage-comparison questions.

Prompts:

- What does aBSREL test that BUSTED does not?
- How many branches were tested, and why does that make multiple-testing correction necessary?
- Which branches, if any, remain significant after correction?
- Are significant branches terminal branches, internal branches, or a mixture? How does that affect interpretation?
- What biological story could explain selection on those branches?
- How would explicit branch labels change the questions you could ask with this dataset?


In [ ]:
summary2 = pd.read_csv(TABLES / 'session2_branch_lineage_summary_long.csv')
summary2


## Session 3: Site-Level Selection

Run:

```bash
bash scripts/03_run_site_level.sh
python scripts/07_collect_results.py
```

In this session you will move from gene-wide and branch-level tests to codon-level evidence. FEL, MEME, and SLAC all summarize selection by site, but they are not asking identical questions.

FEL looks for pervasive selection, meaning a site has an elevated or reduced nonsynonymous rate across the branches being tested. MEME looks for episodic diversifying selection, meaning a site may have experienced positive selection on only a subset of branches. SLAC is a counting-based method that provides a complementary site-level scan.

Treat the site table as a map of candidate codons, not as final biological proof. Sites can be sensitive to alignment quality, recombination, model assumptions, and the number of sequences in the dataset.

Prompts:

- What is the difference between pervasive selection and episodic diversifying selection?
- Which codons were detected by FEL, and do they suggest purifying or diversifying selection?
- Which codons were detected by MEME, and are they the same codons FEL found?
- Which codons were detected by SLAC, and do they agree with FEL or MEME?
- Where do the site-level methods disagree, and what biological or statistical explanation could account for the difference?
- If you had a protein structure or domain map, which sites would you annotate first?
- What would you check in the alignment before trusting a surprising site-level result?


In [ ]:
sites = pd.read_csv(TABLES / 'session3_site_level_tables.csv')
sites.head()

In [ ]:
for method in ['fel', 'meme', 'slac']:
    path = FIGURES / f'session3_site_level_{method}.png'
    if path.exists():
        plt.figure(figsize=(8, 3))
        plt.imshow(plt.imread(path))
        plt.axis('off')
        plt.show()


## Mini-Report Template

Use this final prompt to turn the outputs into a short scientific interpretation. The goal is not to list every p-value. The goal is to state what the analyses suggest, how confident you are, and what would make the conclusion stronger.

- Dataset: Which empirical dataset did you analyze, and why is it biologically interesting?
- Gene-wide result: Did BUSTED support episodic diversifying selection? Which model variant would you emphasize?
- Site-level result: Which codons are the strongest candidates, and do different methods agree?
- Branch-level result: Did aBSREL identify any branches after correction? What might those branches represent?
- Biological interpretation: What is the most plausible selection story supported by these results?
- Caveats: What alignment, sampling, model, or power issues could affect the conclusion?
- Next steps: What analysis, visualization, validation, or additional data would you want next?
